# Loading pdf files and Vectorize the Conent

In [1]:
from pypdf import PdfReader
import os
from tqdm import tqdm
from pymed import PubMed
import json
import pytesseract
from pdf2image import convert_from_path

In [ ]:
class PMCDocLoader:
    def __init__(self, data_dir, save_to):
        self.doc_content=[]
        self.data_dir = data_dir
        self.pubmed = PubMed(tool="MyTool", email="qzhao9@lakeheadu.ca")
        self.save_to = save_to
        self.min_words_per_line = 10
        self.pbar = None

    def get_pmc_metadata(self, pmc_ids):
        """
        Query document meta data
        """
        ids = ','.join([id.lstrip('PMC') for id in pmc_ids])
        try:
            results = self.pubmed._get("/entrez/eutils/esummary.fcgi",parameters={'db':'pmc','id':ids,'retmode':'json'})
            meta_result = results['result']
            uuids = meta_result['uids']
            metadata = {}
            for uuid in uuids:
                result = meta_result[uuid]
                authors = ', '.join([d['name'] for d in result['authors']])
                dois = [id['value'] for id in result['articleids'] if id['idtype']=='doi']
                if any(dois):
                    doi = dois[0]
                dois = '; '.join([f"{id['idtype'].upper()}:{id['value']}" for id in result['articleids']])
                cite_nlm = f"{authors}. {result['title']}. {result['source']}. {result['printpubdate']}; {result['volume']}({result['issue']}):{result['pages']}. Epub {result['epubdate']}. {dois}."
                # print(cite_nlm)
                metadata[f'PMC{uuid}'] = {
                                'id':f'PMC{uuid}',
                                'authors':authors,
                                'title': result['title'],
                                'doi':doi,
                                'cite':cite_nlm}
        except Exception as e:
            print('Failed to get meta data, error: ', e)
            return None

        return metadata


    def load(self, load_meta=True, save_output=True, limited_count=None, load_mode='pdf'):
        file_folders = os.listdir(self.data_dir)
        extracted_docs = []
        steps = 200
        file_folders = [folder for folder in file_folders if os.path.isdir(os.path.join(self.data_dir,folder))]
        
        if limited_count:
            file_folders = file_folders[:limited_count]

        self.pbar = tqdm(range(len(file_folders)))    
        load_fun = self.load_pdf_content
        if load_mode == 'ocr':
            load_fun = self.load_pdf_content_ocr

        for i in range(0, len(file_folders), steps):
            batch = file_folders[i:i+steps]
            if load_meta:
                metadata = self.get_pmc_metadata(batch)
            for folder in tqdm(batch):
                try:
                    self.pbar.set_postfix({'Doc':folder})
                    full_path = os.path.join(self.data_dir,folder)
                    files = os.listdir(full_path)
                    num_of_pdf = len([f for f in files if f.endswith('.pdf') ])
                    if num_of_pdf == 0:
                        # print(folder,'No pdf file.')
                        continue

                    nxml_file = [f for f in files if f.endswith('.nxml')]
                    if not any(nxml_file):
                        continue
                    pdf_file = nxml_file[0].replace('.nxml','.pdf')
                    ful_pdf = os.path.join(full_path,pdf_file)
                    if os.path.isfile(ful_pdf):
                        page_content = load_fun(ful_pdf)
                        if not page_content:
                            continue
                        if load_meta:
                            meta = metadata[folder]
                        else:
                            meta = {'id':folder}
                        meta['content'] = page_content
                        extracted_docs.append(meta)
                finally:
                    self.pbar.update(1)

        self.doc_content = extracted_docs
        
        # Save extracted data
        if save_output:
            with open(self.save_to,'w+') as file:
                json.dump(extracted_docs,file)
        return True
    
    def load_pdf_content(self, pdf_file):
        try:
            reader = PdfReader(pdf_file)

            total_pages = len(reader.pages)
            # print('Total Pages:',total_pages)

            # print('Page 1:',page_1.extract_text())
            pages = []
            for p in range(total_pages):
                page = reader.get_page(p)
                page_content = page.extract_text(space_width=250)
                page_content, is_reference = self.post_extract_processing(page_content)
                pages.append(page_content)
                if is_reference:
                    break
        except:
            print('Failed to load ', pdf_file)
            return None

        return pages
    
    def load_pdf_content_ocr(self, pdf_file):
        """
        Read pdf text using OCR.
        """
        try:
            pages = convert_from_path(pdf_file, 600)
            # print('Pages: ', len(pages))
            # extract text
            pages_text = []
            parent_dir_name = os.path.basename(os.path.dirname(pdf_file))
            for i, p in enumerate(pages):
                self.pbar.set_postfix({'Doc':parent_dir_name,'Page':f'{i+1}/{len(pages)}'})
                page_content = pytesseract.image_to_string(p)
                page_content, is_reference = self.post_extract_processing(i, page_content)
                if page_content:
                    pages_text.append(page_content)
                if is_reference:
                    break
        except Exception as e:
            print('Failed to load ', pdf_file, e)
            return None
        return pages_text

    def post_extract_processing(self,page_number, page_content):
        """
        Clean up extracted text
        """
        lines = page_content.split('\n')
        # print(len(lines))
        new_lines = []
        is_reference = False
        start_of_content = False
        for i in range(len(lines)):
            # print(len(lines[i]))
            if new_lines:
                line_1 = new_lines[-1]
            else:
                line_1 = ''
            line_2 = lines[i]
            if page_number == 0 and not start_of_content:
                if not self.is_start_of_text(line_2):
                    continue
                else:
                    start_of_content = True

            if self.is_reference(line_2):
                # print('is_reference: True')
                is_reference = True
                break
            last_of_line_1 = line_1[-1] if line_1 else ''
            if last_of_line_1 in ['-',',',' ',';','"'] or last_of_line_1.isalpha():
                line_1 = line_1[:-1] if line_1[-1] == '-' else line_1
                new_lines[-1] = line_1 +(' ' if last_of_line_1.isalpha else '') +  line_2
            else:
                if not self.is_ignore(line_2):
                    new_lines.append(line_2)
                else:
                    pass
        return new_lines , is_reference #Paragraph
    
    def is_start_of_text(self,line):
        if line.startswith('Abstract'):
            return True
        
        return False
    
    def is_reference(self, line):
        if line.strip() == 'References':
            return True
        return False
    
    def is_ignore(self,line):
        if not line.strip():
            return True 
        if line.startswith('Acknowledgments') or\
              line.startswith('Conflicts of Interest') or\
              line.startswith('Author Contributions') or\
              line.startswith('Informed Consent Statement') or\
              line.startswith('Data Availability Statement'):
            return True
        # if len(line.strip().split()) < self.min_words_per_line:
        #     return True
        digits=[c for c in line if c.isdigit()]
        if len(digits) / len(line) >= 0.5:
            return True # Most of the charactors are numbers
        
        return False

In [31]:
data_dir = r'/home/qzhao9/datasets/PMC-Papers/files'
metadata_file = '/home/qzhao9/datasets/metadata.jsonl'

doc_loader = PMCDocLoader(data_dir,metadata_file)

doc_loader.load(load_meta=True, save_output=True, load_mode='ocr', limited_count=3)

print(len(doc_loader.doc_content))


100%|██████████| 3/3 [02:21<00:00, 47.02s/it, Doc=PMC7998190, Page=1/16]



100%|██████████| 3/3 [01:33<00:00, 31.22s/it]

2


In [32]:
pages = doc_loader.doc_content[0]['content']
lines_p1 = pages[0]
lines_p_end = pages[-1]
print(doc_loader.doc_content[0]['id'])
for l in lines_p1:
    print(l)

print('=='*30)
if not lines_p_end:
    print('empty page')
for l in lines_p_end:
    print(l)


PMC11781193
Abstract  Inflammatory diseases of the human gastrointestinal tract are affected by the microbes that reside in the mucosal surfaces. Pa tients with inflammatory bowel diseases (IBD) have altered bacterial and fungal intestinal compositions, including higher levels of fecal Candida yeasts. Ongoing research indicates that genetic and phenotypic diversity of Candida albicans may be linked with disease severity. Here, we set out to investigate feces-derived C. albicans strains from individuals with IBD and healthy volunteers through microsatellite-based genotyping and phenotypic assays. A seven-locus microsatellite panel was applied, of which six loci were newly developed. It appears that there is no specific lineage of C. albicans that is associated with IBD, but rather that the three study popula tions (Crohn’s disease, ulcerative colitis, healthy volunteers) do have distinguishable distributions of genotypes. In addition, pheno typic characterization by means of enzyme rele

# Load documents and build vector DB

In [6]:
import json


metadata_file = '/home/qzhao9/datasets/metadata.jsonl'
with open(metadata_file,'r+') as docfile:
    content_of_docs = json.load(docfile)

print('document loaded.')


document loaded.


In [ ]:
content_of_docs[0]['content']

In [7]:
[c for c in content_of_docs[0]['content'][-2] if len(c.split()) < 10]

[]

## ChromaDB

In [ ]:
# import chromadb
# chroma_client = chromadb.PersistentClient('/home/qzhao9/datasets/chromadb')
# collection = chroma_client.get_or_create_collection(name="my_collection")


In [8]:
doc = content_of_docs[0]
doc_id = doc['id']
pages = doc['content']
page_ids = [f'{doc_id}_P{i+1}' for i in range(len(pages))]
print(page_ids)
print(pages[0])
for pid, paragraphs in enumerate(pages):
    ids = [f'{page_ids[pid]}_{i}' for i in range(len(paragraphs))]
    print(ids)

['PMC11781193_P1', 'PMC11781193_P2', 'PMC11781193_P3', 'PMC11781193_P4', 'PMC11781193_P5', 'PMC11781193_P6', 'PMC11781193_P7', 'PMC11781193_P8', 'PMC11781193_P9']
['Isabelle A.M. van Thiel ©1223," Irini A.M. Kreulen ©23 T Mélanie V. Bénard ©3414, Marcus C. de Goffau O25 Bart Theelen 1, Sigrid  E.M. Heinsbroek ©2234 Patrycja K. Zylka\', Cyriel Y. Ponsioen ©. Teun Boekhout 1, Wouter J. de Jonge ©2346 Sgren Rosendahl o7', 'tWesterdijk Fungal Biodiversity Institute, Royal Dutch Academy of Arts and Sciences, Uppsalalaan 8, 3584 CT Utrecht, The Netherlands  *Tytgat Institute for Liver and Intestinal Research, Amsterdam University Medical Centers, Location Academic Medical Center (AMC), Meibergdreef 69-71, 1105 BK Amsterdam, The Netherlands  >Amsterdam UMC, University of Amsterdam, Amsterdam Gastroenterology Endocrinology Metabolism, Meibergdreef 9, 1105 AZ Amsterdam, The Netherlands “Department of Gastroenterology and Hepatology, Amsterdam University Medical Centers, Location Academic Medica

## documents

In [ ]:
import chromadb
from chromadb.utils.data_loaders import ImageLoader
from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction

client = chromadb.PersistentClient('/home/qzhao9/datasets/chromadb')

data_loader = ImageLoader()
embedding_function = OpenCLIPEmbeddingFunction()

coll_name='multimodal_collection'
client.delete_collection(name=coll_name)
collection = client.get_or_create_collection(
    name=coll_name,
    embedding_function=embedding_function,
    data_loader=data_loader
)

In [15]:
content_of_docs[1]['content']

[["Jennifer Joan Ryan '*t®, Andrea Monteagudo-Mera 7, Nikhat Contractor > and Glenn R. Gibson 2",
  ': Helfgott Research Institute, National University of Natural Medicine, Portland, OR 97201, USA Department of Food and Nutritional Sciences, University of Reading, Reading RG6 6AD, UK; a.monteagudo@reading.ac.uk (A.M.-M.); g.r.gibson@reading.ac.uk (G.R.G.)',
  "Abstract: Intestinal dysbiosis has been described in patients with certain gastrointestinal conditions including irritable bowel syndrome (IBS) and ulcerative colitis. 2’-fucosyllactose (2’-FL), a prebiotic human milk oligosaccharide, is considered bifidogenic and butyrogenic. To assess prebiotic effects of 2’-FL, alone or in combination with probiotic strains (potential synbiotics), in vitro experiments were conducted on stool from healthy, IBS, and ulcerative colitis adult donors. In anaerobic batch culture fermenters, Bifidobacterium and Eubacterium rectale-Clostridium coccoides counts, and short-chain fatty acids (SCFAs) incl

In [10]:
# from PIL import Image
# import numpy as np
from tqdm import tqdm

for doc in tqdm(content_of_docs[:]):
    doc_id = doc['id']
    pages = doc['content']
    page_ids = [f'{doc_id}_P{i+1}' for i in range(len(pages))]
    for pid, paragraphs in enumerate(pages):
        ids = [f'{page_ids[pid]}_{i}' for i in range(len(paragraphs))]
        metadatas=[{'title':doc['title'],'page':pid,'cite':doc['cite']} for i in range(len(paragraphs))]
        collection.add(
            ids=ids,
            documents=paragraphs,
            metadatas=metadatas
        )

# def load_image(uri):
#     img = Image.open(uri)
#     img_np = np.array(img, dtype=np.uint8, copy=True)
#     return img_np

# collection.add(
#     ids=["id1", "id2"],
#     images=[load_image("/home/qzhao9/datasets/kvasir-dataset/polyps/0a874a44-90ea-42ea-a467-92a1d3a4a26c.jpg"), 
#           load_image("/home/qzhao9/datasets/kvasir-dataset/ulcerative-colitis/6acfb44a-2b22-4d49-a3bf-9ffc2461c3f9.jpg")]
# )

# # collection.add(
# #     ids=["id3", "id4"],
# #     documents=["This is a polyp image", "This is a UC image"]
# # )




 50%|█████     | 1/2 [00:50<00:50, 50.67s/it]


ValueError: Non-empty lists are required for ['ids', 'metadatas', 'documents'] in add.

In [ ]:
print(client.heartbeat())
print(collection.count())

In [ ]:
content_of_docs[2]['content'][0]

In [ ]:
results = collection.query(query_texts='ileal pouch-anal anastomosis surgery', n_results=3)
print(results['documents'])
print(results['ids'])
print(results['metadatas'])
print(results['distances'])

# Build FAISS index

In [ ]:
# from sentence_transformers import SentenceTransformer
# import faiss
# import numpy as np

# # 1. Load embedding model
# embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# # 2. Compute embeddings
# embeddings = embedder.encode(corpus[:3297], convert_to_numpy=True, show_progress_bar=True)
# print('Done computing embeddings')
# # 3. Build FAISS index
# dimension = embeddings.shape[1]
# print('dimension=',dimension)
# index = faiss.IndexFlatL2(dimension)
# index.add(embeddings)
# print('Done adding embeddings to index')


In [ ]:
def pico_prompt(question):
    template = f"""
You are a clinical evidence extraction assistant.
Rewrite the following medical question into structured PICO format:

P: Patient / Population  
I: Intervention  
C: Comparison  
O: Outcome  

Question: {question}

Return only structured PICO elements.
"""
    return template


def retrieve_top_k(pico_query, k=3):
    q_emb = embedder.encode([pico_query], convert_to_numpy=True)
    distances, indices = index.search(q_emb, k)
    
    retrieved = [corpus[i] for i in indices[0]]
    return retrieved

def medical_rag_qa(question,generator):
    # Step 1 — Convert to PICO
    pico_query = pico_prompt(question)

    # Step 2 — Retrieve evidence
    retrieved = retrieve_top_k(pico_query, k=3)
    context = "\n".join(retrieved)
    print('Context:', context)
    # Step 3 — Construct final answer prompt
    final_prompt = f"""
You are a medical expert. Use ONLY the evidence provided.

=== Evidence ===
{context}

=== Question ===
{question}

Provide:
1. A concise medical answer
2. A rationale referencing the evidence
3. Confidence level (low / medium / high)
"""

    # Step 4 — Generate final answer
    result = generator(question=question,     
                       context='context')
    answer = result['answer']
    print(result['score'])
    return answer



In [ ]:
from transformers import pipeline, AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained(
#     "distilbert-base-cased-distilled-squad",
#     use_fast=True
# )

generator = pipeline("question-answering", 
                        model='distilbert-base-cased-distilled-squad', 
                        # tokenizer=tokenizer,
                        # use_fast=True
                        )

In [ ]:
print(type(generator.tokenizer))
print(generator.tokenizer.is_fast)
print(type(generator.model))

import transformers
import inspect

print(transformers.__version__)
print(inspect.getfile(transformers))



In [ ]:
generator(question="What is a good example of a question answering dataset?",  context='Peter the cat is a good example')

In [ ]:
question = """
A few non-lift polyps were found in a 55-year-old patient. What could be the problem?
"""

print(medical_rag_qa(question, generator))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, json

model_name='Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)



In [ ]:
def pico_query(user_query):
    prompt = pico_prompt(user_query)
    inputs = tokenizer(prompt, return_tensors="pt")
    tokens = model.generate(**inputs, max_length=512)
    output = tokenizer.decode(tokens[0], skip_special_tokens=True)
    return output #json.loads(output[output.find("{"):output.rfind("}")+1])

In [ ]:
pico_query(question)

In [ ]:
# meta = reader.metadata

# # All the following could be None!
# print('Title:',meta.title)
# print('Author:',meta.author)
# print('Subject:',meta.subject)
# print('keywords:',meta.keywords)
# # print(meta.producer)
# print('Created On:',meta.creation_date)
# print(meta.modification_date)

# pdfplumber

In [ ]:
import pdfplumber
import os

pdf_file = '/home/qzhao9/datasets/PMC-Papers/files/PMC11781193/ftaf001.pdf'
print()
# with pdfplumber.open(pdf_file) as pdf:
#     first_page = pdf.pages[1]
#     text = first_page.extract_text(space_width=200)
#     table = first_page.extract_table()
#     image = first_page.images
#     print(text) 
#     print(table)
#     print(image)

# from PyPDF2 import PdfReader

# reader = PdfReader(pdf_file)
# page = reader.pages[1]
# print(page.extract_text(space_width=200))

# doc_loader.load_pdf_content_ocr(pdf_file)





In [ ]:
pages = convert_from_path(pdf_file, 600)
print('Pages: ', len(pages))
# extract text
pages = []
for p in pages:
    page_content = pytesseract.image_to_string(p)
    print('Extracted :',len(page_content))
    page_content, is_reference = doc_loader.post_extract_processing(page_content)
    pages.append(page_content)
    if is_reference:
        break


In [ ]:
pages

In [ ]:
text_data.split('\n')

# "unstructured[all-docs]"

In [ ]:
# from unstructured.partition.auto import partition
# blocks = partition(filename=pdf_file)
# for block in blocks:
#     print(f"{block.category}: {block.text}")